# Chapter 1 Exercises 2 and 3: Straight-Section FODO Cells

This notebook solves the forward and reverse straight-section FODO exercises from Chapter 1.

The arc bends are replaced by straight-section drifts `DB` of length 5.855 m. We first optimize the forward straight FODO cell to 90 degrees in both transverse planes, then verify that the reverse straight FODO cell has the same optimized strengths.

In [1]:
using SciBmad
using GTPSA
using LinearAlgebra
using Printf

function find_tutorial_root()
    candidates = [
        pwd(),
        normpath(joinpath(pwd(), "..")),
        normpath(joinpath(pwd(), "..", "..")),
        normpath(joinpath(pwd(), "..", "..", "..")),
    ]
    for root in candidates
        if isfile(joinpath(root, "chapter01_fodo_scibmad.ipynb")) && isdir(joinpath(root, "lattices", "chapter_1"))
            return root
        end
    end
    error("Cannot locate Ring_Design_Tutorial_SciBmad root. Run this notebook from the tutorial folder, lattices/chapter_1, or lattices/chapter_1/exercises.")
end

const tutorial_root = find_tutorial_root()

const L_quad = 0.5
const D1_len = 0.609
const D2_len = 1.241
const DB_len = 5.855

const species_ref = Species("electron")
const E_ref = 18e9

D1() = Drift(L=D1_len)
D2() = Drift(L=D2_len)
DB() = Drift(L=DB_len)


DB (generic function with 1 method)

## Build the Straight FODO Cells

The forward straight FODO has the sequence

$$
(QFSS, D1, DB, D2, QDSS, D1, DB, D2),
$$

while the reverse straight FODO has the drift order reversed around the same quadrupoles:

$$
(QFSS, D2, DB, D1, QDSS, D2, DB, D1).
$$


In [2]:
function build_forward_straight_fodo(kqf, kqd)
    return Beamline(
        [
            Quadrupole(name="QFSS", L=L_quad, Kn1=kqf), D1(), DB(), D2(),
            Quadrupole(name="QDSS", L=L_quad, Kn1=kqd), D1(), DB(), D2(),
        ];
        species_ref=species_ref,
        E_ref=E_ref,
    )
end

function build_reverse_straight_fodo(kqf, kqd)
    return Beamline(
        [
            Quadrupole(name="QFSS", L=L_quad, Kn1=kqf), D2(), DB(), D1(),
            Quadrupole(name="QDSS", L=L_quad, Kn1=kqd), D2(), DB(), D1(),
        ];
        species_ref=species_ref,
        E_ref=E_ref,
    )
end


build_reverse_straight_fodo (generic function with 1 method)

## GTPSA Transfer Map and Optimizer

The two optimized variables are the strengths of `QFSS` and `QDSS`. We add them as GTPSA descriptor parameters during lattice construction. The residual is formed from the constant terms of the phase advances, while the Jacobian used by the optimizer is read from the parameter gradients.

In [3]:
function track_a_particle(v0, beamline)
    v = similar(v0)
    v .= v0
    bunch = Bunch(v; species=beamline.species_ref, p_over_q_ref=beamline.p_over_q_ref)
    track!(bunch, beamline)
    return copy(bunch.coords.v)
end

function linear_map_with_descriptor(beamline, d; x0=zeros(6))
    xvars = vars(d)
    v0 = [x0[i] + xvars[i] for i in 1:6]
    vout = track_a_particle(v0, beamline)

    M = Matrix{Any}(undef, 6, 6)
    for i in 1:6, j in 1:6
        powers = zeros(Int, 6)
        powers[j] = 1
        M[i, j] = par(vout[i], [powers...,:])
    end
    return M
end

parameter_gradient(x) = GTPSA.gradient(x, include_params=true)[7:end]
tps_const(x) = try x[zeros(Int, 6)] catch; x end

function stable_phase_advance(M2)
    cos_mu = clamp(0.5 * tr(M2), -1.0, 1.0)
    return acos(cos_mu)
end

function stable_phase_advance_with_params(M2)
    cos_mu = 0.5 * tr(M2)
    return acos(cos_mu)
end

function periodic_twiss_from_map(M2)
    mu = stable_phase_advance(M2)
    smu = sin(mu)
    beta = M2[1, 2] / smu
    alpha = (M2[1, 1] - M2[2, 2]) / (2smu)
    return (beta=beta, alpha=alpha, mu=mu)
end


periodic_twiss_from_map (generic function with 1 method)

In [4]:
const d_knobs = Descriptor(6, 2, 2, 1)
const dk = params(d_knobs)

function phase_advances_with_knobs(k, builder)
    M = linear_map_with_descriptor(builder(k[1] + dk[1], k[2] + dk[2]), d_knobs)
    mu_x = stable_phase_advance_with_params(M[1:2, 1:2])
    mu_y = stable_phase_advance_with_params(M[3:4, 3:4])
    return mu_x, mu_y
end

function phase_residual(k, builder)
    mu_x, mu_y = phase_advances_with_knobs(k, builder)
    return [tps_const(mu_x) - pi / 2, tps_const(mu_y) - pi / 2]
end

function phase_jacobian(k, builder)
    mu_x, mu_y = phase_advances_with_knobs(k, builder)
    return vcat(parameter_gradient(mu_x)', parameter_gradient(mu_y)')
end

function damped_least_squares(f, jacobian, x0; maxiter=40, tol=1e-13, lambda0=1e-3)
    x = copy(x0)
    lambda = lambda0

    for iter in 1:maxiter
        r = f(x)
        merit_now = sum(abs2, r)
        J = jacobian(x)
        step = -(J' * J + lambda * I) \ (J' * r)
        trial = x + step
        merit_trial = sum(abs2, f(trial))

        @printf("iter %2d  merit = %.6e  step = %.3e  lambda = %.3e\n", iter, merit_now, norm(step), lambda)

        if merit_trial < merit_now
            x = trial
            lambda /= 10
        else
            lambda *= 10
        end

        if norm(r) < tol || norm(step) < tol
            break
        end
    end
    return x
end


damped_least_squares (generic function with 1 method)

## Exercise 2: Forward Straight-Section FODO

Optimize the forward straight-section FODO cell so that both transverse phase advances are 90 degrees.

In [5]:
forward_residual(k) = phase_residual(k, build_forward_straight_fodo)
forward_jacobian(k) = phase_jacobian(k, build_forward_straight_fodo)

k_start = [0.35, -0.35]
k_forward = damped_least_squares(forward_residual, forward_jacobian, k_start)

kQFSS = k_forward[1]
kQDSS = k_forward[2]
K_ss = 0.5 * (kQFSS - kQDSS)

M_forward = tps_const.(linear_map_with_descriptor(build_forward_straight_fodo(kQFSS, kQDSS), d_knobs))
mu_x_forward = stable_phase_advance(M_forward[1:2, 1:2])
mu_y_forward = stable_phase_advance(M_forward[3:4, 3:4])
twiss_x_forward = periodic_twiss_from_map(M_forward[1:2, 1:2])
twiss_y_forward = periodic_twiss_from_map(M_forward[3:4, 3:4])

println("\nOptimized forward straight FODO:")
@printf("  kQFSS = %.15f\n", kQFSS)
@printf("  kQDSS = %.15f\n", kQDSS)
@printf("  K_ss  = %.15f\n", K_ss)
@printf("  mu_x  = %.12f deg\n", rad2deg(mu_x_forward))
@printf("  mu_y  = %.12f deg\n", rad2deg(mu_y_forward))
println("  final residual = ", forward_residual(k_forward))


iter  1  merit = 2.460485e-04  step = 2.776e-03  lambda = 1.000e-03
iter  2  merit = 1.834418e-09  step = 7.538e-06  lambda = 1.000e-04
iter  3  merit = 2.087214e-19  step = 8.040e-11  lambda = 1.000e-05
iter  4  merit = 3.352659e-30  step = 2.547e-16  lambda = 1.000e-06

Optimized forward straight FODO:
  kQFSS = 0.351957452649287
  kQDSS = -0.351957452649286
  K_ss  = 0.351957452649287
  mu_x  = 90.000000000000 deg
  mu_y  = 90.000000000000 deg
  final residual = [4.440892098500626e-16, -1.7763568394002505e-15]


## Exercise 3: Reverse Straight-Section FODO

The reverse straight-section FODO uses the same quadrupole strengths. Only the drift order changes. We verify the 90-degree phase advance directly.

In [6]:
M_reverse = tps_const.(linear_map_with_descriptor(build_reverse_straight_fodo(kQFSS, kQDSS), d_knobs))
mu_x_reverse = stable_phase_advance(M_reverse[1:2, 1:2])
mu_y_reverse = stable_phase_advance(M_reverse[3:4, 3:4])
twiss_x_reverse = periodic_twiss_from_map(M_reverse[1:2, 1:2])
twiss_y_reverse = periodic_twiss_from_map(M_reverse[3:4, 3:4])

println("\nReverse straight FODO check using the same strengths:")
@printf("  mu_x  = %.12f deg\n", rad2deg(mu_x_reverse))
@printf("  mu_y  = %.12f deg\n", rad2deg(mu_y_reverse))
println("  residual = ", phase_residual([kQFSS, kQDSS], build_reverse_straight_fodo))

println("\nForward straight-section periodic Twiss:")
@printf("  beta_x  = %.10f, alpha_x = %.10f\n", twiss_x_forward.beta, twiss_x_forward.alpha)
@printf("  beta_y  = %.10f, alpha_y = %.10f\n", twiss_y_forward.beta, twiss_y_forward.alpha)

println("\nReverse straight-section periodic Twiss:")
@printf("  beta_x  = %.10f, alpha_x = %.10f\n", twiss_x_reverse.beta, twiss_x_reverse.alpha)
@printf("  beta_y  = %.10f, alpha_y = %.10f\n", twiss_y_reverse.beta, twiss_y_reverse.alpha)



Reverse straight FODO check using the same strengths:
  mu_x  = 90.000000000000 deg
  mu_y  = 90.000000000000 deg
  residual = [4.440892098500626e-16, -2.4424906541753444e-15]

Forward straight-section periodic Twiss:
  beta_x  = 27.2059882001, alpha_x = -2.4024903657
  beta_y  = 4.9609141145, alpha_y = 0.4846055571

Reverse straight-section periodic Twiss:
  beta_x  = 27.2059882001, alpha_x = -2.4024903657
  beta_y  = 4.9609141145, alpha_y = 0.4846055571


## Save the Result

The saved solution file can be loaded by later chapters instead of hard-coding the straight-section FODO strength.

In [7]:
solution_text = """
# chapter1_fodoSS_solution.jl
# Auto-generated by chapter1_straight_fodo_solution.ipynb.

K_ss = $(repr(K_ss))
kQFSS = $(repr(kQFSS))
kQDSS = $(repr(kQDSS))

straight_twiss_x = (beta=$(repr(twiss_x_forward.beta)), alpha=$(repr(twiss_x_forward.alpha)))
straight_twiss_y = (beta=$(repr(twiss_y_forward.beta)), alpha=$(repr(twiss_y_forward.alpha)))
reverse_straight_twiss_x = (beta=$(repr(twiss_x_reverse.beta)), alpha=$(repr(twiss_x_reverse.alpha)))
reverse_straight_twiss_y = (beta=$(repr(twiss_y_reverse.beta)), alpha=$(repr(twiss_y_reverse.alpha)))
"""

solution_path = joinpath(tutorial_root, "lattices", "chapter_1", "chapter1_fodoSS_solution.jl")
mkpath(dirname(solution_path))
write(solution_path, solution_text)

println("Wrote: ", solution_path)
println()
println(solution_text)


Wrote: D:\Onedrive\Cornell\大一\Accelerator Physics\Ring_Design_Development\Ring_Design_Tutorial_SciBmad\lattices\chapter_1\chapter1_fodoSS_solution.jl

# chapter1_fodoSS_solution.jl
# Auto-generated by chapter1_straight_fodo_solution.ipynb.

K_ss = 0.3519574526492866
kQFSS = 0.35195745264928663
kQDSS = -0.35195745264928646

straight_twiss_x = (beta=27.2059882001124, alpha=-2.402490365720772)
straight_twiss_y = (beta=4.960914114526318, alpha=0.48460555713271297)
reverse_straight_twiss_x = (beta=27.2059882001124, alpha=-2.402490365720772)
reverse_straight_twiss_y = (beta=4.9609141145263225, alpha=0.4846055571327126)

